# N096 · 堆贪心与撤销局部决策

**目标：** 在资源受限时替换已选集合中的最差元素。

**先修：** N057, N061, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 截止期排序；最大堆替换；可用项目；预算阈值；交换证明。

**配套讲解来源：** [同名逐章意见](../../comment/096_greedy_heap_replacement.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

普通的贪心是"选了就不回头"，这章的贪心会**反悔**：允许把已经选进集合里的"最差的那个"踢出去，换成更好的。堆（优先队列）就是干这件事的家伙。

**问题一（课程表 III，LeetCode 630）**：每门课 `[时长, 截止日]`，课程从第 1 天开始、连续上 `时长` 天、必须在第 `截止日` 之前（含当天，即累计时长不超过截止日）上完，同一时间只能上一门课。问最多能上几门。给你具体数字（“运行示例”部分 实例）：`[[100,200],[200,1300],[1000,1250],[2000,3200]]`。**答案是 3 门**。为什么？按截止日排序后依次考虑：上 100（100≤200）；上 1000（累计 1100≤1250）；上 200（累计 1300≤1300，贴线通过）；轮到 2000 时累计会变成 3300>3200，塞不下——这时算法把已选集合里最长的那门（2000 本身）踢掉，剩下 {100,1000,200} 三门总时长 1300，全部赶得上截止日。注意巧妙之处：最后那门 2000 是"先选进来又被踢出去"，不反悔的话你就只剩硬生生放弃。

**问题二（IPO，LeetCode 502）**：你有启动资金 `w`、最多做 `k` 个项目，项目 i 需要门槛资本 `capital[i]` 才能启动，做完净赚 `profits[i]`。问最终资本最多变成多少。具体数字：`k=2, w=0, profits=[1,2,3], capital=[0,1,1]`。**答案是 4**：第一步只有项目 0（门槛 0）可做，赚 1 变成 1；第二步门槛 1 的两个项目都解锁了，挑利润最大的 3 做，变成 4。

## 2. 基础知识：先读懂这些词

- **堆贪心（heap greedy）**：用堆维护"当前已选集合"，当新元素塞不下时，快速找到并删除已选中最差的一个。它的本质能力是**撤销先前的局部决策**，这弥补了普通贪心"一条路走到黑"的缺陷。
- **Python 的 heapq 与负号技巧**：Python 的 `heapq` 只提供小根堆（堆顶是最小值）。想要"大根堆"（堆顶是最大值），就把数取负存进去——`heappush(chosen,-duration)` 存的是负时长，堆顶 `-chosen[0]` 就是最大时长。本章两个函数都用这招。
- **截止期排序（sort by deadline）**：课程题先按截止日从早到晚排序再决策。直觉是"deadline 早的先安排，晚的随时能补"，并且可以证明这样排序不会丢掉任何可行方案（见第五节）。
- **撤销 / 替换（revocation / replacement）**：课程题的替换规则是"塞不下时删掉已选里时长最长的一门"。数量没变（都是这些门数），但总占用时间变小，给未来留出更大余地。
- **可用项目（available projects）**：IPO 题里，当前资本 `w` 能解锁的那些项目组成一个集合。资本只会增加，所以项目只会不断解锁、集合只增不减——用一个堆维护"当前可做项目里利润最大的"。
- **资本门槛（capital threshold）**：项目的启动资格线。只有 `capital[i] <= w` 时项目 i 才进入可选池。注意利润是"净增加"，不会倒扣你的门槛资本。
- **交换证明（exchange proof）**：又一次登场的老朋友——把最优方案的第一步换成贪心的第一步，论证不会更差。本章两道题的正确性都靠它。
- **预算阈值（非正收益提前停）**：IPO 的一个防御细节：如果可选池里利润最大的项目收益都不超过 0，做了也白做（甚至亏），本章实现选择直接提前结束。

## 3. 思路推导：从小例子开始

**课程表的思路：**

1. **Step 1：按截止日排序。** 截止早的课程必须先被考虑，否则会被后面长课程挤掉机会。
2. **Step 2：逐门课程"先斩后奏"。** 无条件把当前课程压入堆（记它的负时长），累计时长 `total` 加上它的时长。
3. **Step 3：如果 `total` 超过了当前截止日，就反悔。** 从堆里弹出时长最大的一门（可能是刚压进去的这门自己，也可能是更早的老课程），把它的时长从 `total` 里扣掉。这样已选课程数尽可能多、总时长尽可能小。
4. **Step 4：扫完所有课程后，堆里剩几门就是答案。**

**手算演示（“运行示例”部分 的 `[[100,200],[200,1300],[1000,1250],[2000,3200]]`，按截止日排序后顺序是 100→1000→200→2000）：**

| 时长 | 截止 | 撤销 | 累计时长 | 课程数 |
|---|---|---|---|---|
| 100 | 200 | 无 | 100 | 1 |
| 1000 | 1250 | 无 | 1100 | 2 |
| 200 | 1300 | 无 | 1300 | 3 |
| 2000 | 3200 | 2000 | 1300 | 3 |

这张表就是 “运行示例”部分 运行后打印的内容。前三是"顺利塞入"；第四行 2000 塞入后累计 3300 超了 3200，于是撤销时长最大的 2000，累计时长回到 1300，课程数保持 3。

**IPO 的思路：**

1. **Step 1：项目按门槛资本从小到大排序。** 这样"解锁"就是单调向右推进一个指针。
2. **Step 2：每轮开始，把所有门槛 ≤ 当前资本 `w` 的项目全部压入"可选堆"**（存负利润，堆顶即最大利润）。
3. **Step 3：从可选堆里取利润最大的做**：`w += 利润`。如果堆空（没项目可做）或堆顶利润 ≤ 0（做了不划算），提前结束。
4. **Step 4：最多做 `k` 轮，返回最终 `w`。**

手算（`k=2, w=0, profits=[1,2,3], capital=[0,1,1]`）：排序后项目为 (门槛 0, 利润 1)、(门槛 1, 利润 2)、(门槛 1, 利润 3)。第 1 轮：门槛 ≤0 的只有第一个，入堆，做它，w=0+1=1；第 2 轮：门槛 ≤1 的剩下两个都入堆，堆顶利润 3，做它，w=1+3=4；轮数用完，返回 4。

## 4. 核心代码：schedule_course

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def schedule_course(courses):
    total = 0
    chosen = []
    for duration, deadline in sorted(courses, key=lambda c: c[1]):
        heappush(chosen, -duration)
        total += duration
        if total > deadline:
            total += heappop(chosen)
    return len(chosen)
```

### 逐段读懂代码

“分段实现”部分 从 `heapq` 导入 `heappush` 和 `heappop`，然后是两个函数。

**函数一：最多上几门课（以下是 “分段实现”部分 的逐字代码）**

```python
def schedule_course(courses):
    total = 0
    chosen = []
    for duration, deadline in sorted(courses, key=lambda c: c[1]):
        heappush(chosen, -duration)
        total += duration
        if total > deadline:
            total += heappop(chosen)
    return len(chosen)
```

- `total` 是已选课程的总时长，`chosen` 是存负时长的堆。
- `sorted(courses,key=lambda c:c[1])` 按第二列（截止日）排序，每轮解包出 `duration,deadline`。
- `heappush(chosen,-duration)` 先无条件收下这门课：存负号是为了让"堆顶 = 最长时长的那门"。`total+=duration` 同步记账。
- `if total>deadline: total+=heappop(chosen)` 是最精炼的一行：塞爆了就弹出堆顶（负的最小 = 时长最大那门），弹出的值是 `-最大时长`，所以 `total += (-最大时长)` 实际上就是把那门的时长减掉。为什么不比较"刚进的这门"和"堆里最长的"？因为如果刚进的这门就是最长的，弹掉它等价于"这轮白来、直接放弃"，逻辑自洽；如果是别的老课程更长，弹老留新——两者课程数不变，但留下的组合总时长更小。
- `return len(chosen)`：堆里剩几门就是最多能上几门。

**函数二：IPO（以下是 “分段实现”部分 的逐字代码）**

```python
def find_maximized_capital(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    i = 0
    available = []
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heappush(available, -projects[i][1])
            i += 1
        if not available or available[0] >= 0:
            break
        w -= heappop(available)
    return w
```

- `projects=sorted(zip(capital,profits))`：把门槛和利润配成对并按门槛排序（门槛相同时利润小的在前，`zip` 元组按字典序排的自然结果）。
- `i` 是"下一个待解锁项目"的指针，`available` 是可选堆（存负利润）。
- 内层 `while`：只要还有项目且门槛 ≤ 当前资本 w，就解锁入堆。因为 projects 已排序、w 只增不减，`i` 从不回头，每个项目一生只入堆一次。
- `if not available or available[0]>=0: break`：两个提前结束条件——可选池空了（w 太低，啥都做不了），或者堆顶 `-利润 >= 0` 即最大利润 ≤ 0（再做多赚一分没有，甚至亏钱），继续做毫无意义。这就是 “正确性与复杂度”部分 说的"提前停在非正最大收益处"。
- `w-=heappop(available)`：弹出的又是负利润，减去负数等于加上利润。又是一处负号技巧。
- `return w`：k 轮做完或提前退出后的最终资本。

两个函数的"堆里存什么、堆顶代表谁"对照：

| 对照项 | schedule_course | find_maximized_capital |
|---|---|---|
| 堆中存的是 | `-时长`（已选课程） | `-利润`（已解锁未做项目） |
| 堆顶的含义 | 已选中最长的那门课 | 可做项目中利润最大的 |
| 弹堆顶的时机 | 累计时长超截止（反悔） | 每轮做掉一个项目（消费） |
| 堆顶弹出后干什么 | 从 total 里扣回它的时长 | 加进资本 w，进冷却（下一轮） |
| 提前结束条件 | 无（扫完为止） | 堆空或最大利润 ≤ 0 |

## 5. 分段实现：按顺序运行

**本章接口：** `schedule_course(courses)`、`find_maximized_capital(k, w, profits, capital)`

### 导入本章使用的库

In [1]:
from heapq import heappush, heappop

### schedule_course

In [2]:
def schedule_course(courses):
    total = 0
    chosen = []
    for duration, deadline in sorted(courses, key=lambda c: c[1]):
        heappush(chosen, -duration)
        total += duration
        if total > deadline:
            total += heappop(chosen)
    return len(chosen)

### find_maximized_capital

In [3]:
def find_maximized_capital(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    i = 0
    available = []
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heappush(available, -projects[i][1])
            i += 1
        if not available or available[0] >= 0:
            break
        w -= heappop(available)
    return w

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

courses=[[100,200],[200,1300],[1000,1250],[2000,3200]]; heap=[]; total=0; rows=[]
for duration,deadline in sorted(courses,key=lambda x:x[1]):
    heappush(heap,-duration); total+=duration; removed=None
    if total>deadline: removed=-heappop(heap); total-=removed
    rows.append((duration,deadline,removed,total,len(heap)))
show_table(['时长','截止','撤销','累计时长','课程数'],rows)

时长,截止,撤销,累计时长,课程数
100,200,None,100,1
1000,1250,None,1100,2
200,1300,None,1300,3
2000,3200,2000,1300,3


## 7. 正确性、适用条件与复杂度

课程按截止期交换排序不损害可行性。加入新课程若超时，必须至少放弃一门，删除最长者在保持最多数量的候选中使占用时间最小。IPO利润非负时，先选更大利润不会减少后续可选项目集合，可将最优顺序首项替换为当前最大收益项目。

**代价：** O(n log n+k log n)时间上界，O(n)辅助空间。IPO假定最多选k项、每项只能一次、capital是启动门槛；负收益并非本章标准题域，实现会提前停在非正最大收益处。

### 展开理解

**课程表为什么对？** 论证分两层。

- 第一层（排序无害）：任何一个可行的上课方案，把它内部的课程按截止日重排一遍依然可行——交换相邻两门"先上截止晚的、后上截止早的"课程，不会让任何一门超期（截止早的那门只会更早结束）。所以只考察"按截止日顺序"的方案不丢最优解。
- 第二层（替换规则最优）：处理到某门课时若总时长超了截止日，说明当前集合必须至少放弃一门。我们想要的是"课程数一样多的前提下总时长最小"，那么放弃最长的那门就是最优选择——放弃任何一门都能回到合法，而放弃最长的削减的时长最多。堆顶恰好维护着这个"最长者"，一步到位。
- 两层合起来：每个阶段都维持着"当前考虑过的课程里，数量最多、且同数量下总时长最小"的方案，扫完自然全局最优。

**IPO 为什么对？** 关键在于利润非负时资本 `w` 单调不减。

- 由此推出可选项目集合只会扩大不会缩小——先选利润大的不会让后面"无项目可选"。
- 交换论证：设某个最优选择序列，把它第一步换成"当前可选池中利润最大"的项目，新方案的资本从第二步起不低于原方案（因为第一步赚得不少于原来），于是原方案后面每一步在新资本下依然都做得起，方案仍可行且最终资本不差。反复替换，贪心序列就对齐到最优。
- 注意这套论证依赖利润非负；如果存在负利润项目，"先做大利润"可能配合"做负利润解锁"这类操作，本章实现不覆盖这种题域。

**复杂度。** 分开数：

- 课程表：排序 O(n log n)，n 门课各做一次堆操作、每次 O(log n)，总计 O(n log n)。
- IPO：排序 O(n log n)；k 轮循环里，解锁 while 摊到全程只有 n 次入堆，每轮一次弹出 O(log n)，总计 O(n log n + k log n)。
- 空间：两个函数都开一个不超过 n 大小的堆，是 O(n)。
- 具体感受：n=10⁴ 门课，排序加堆操作大约 10⁴×14 ≈ 1.4×10⁵ 次比较交换量级，瞬间完成；即便 n=10⁵ 也在零点几秒内。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试分两层。

**第一层：四条固定断言。**

- `assert schedule_course([[100,200],[200,1300],[1000,1250],[2000,3200]])==3`：正常值测试，即正文和 “运行示例”部分 手算的例子，锁定"塞不下时撤销 2000"后的答案 3。
- `assert schedule_course([[5,4]])==0`：特殊值测试——单门课时长 5 超过截止 4，物理不可能完成。压入后 total=5>4 触发弹出，堆被弹空，返回 0。它盯防"只顾计数不看合法性"的实现。
- `assert find_maximized_capital(2,0,[1,2,3],[0,1,1])==4`：正常值测试，即正文手算的例子，验证"逐轮解锁 + 挑最大利润"两步后的终值 4。
- `assert find_maximized_capital(3,0,[5],[1])==0`：边界测试——唯一的项目门槛是 1 而启动资金只有 0，一个项目都解锁不了，`available` 为空提前 break，资本原地返回 0。

**第二层：随机对拍暴力。** 两套独立裁判：

- 课程裁判（100 组）：每组随机生成 0~7 门课（时长 1~6、截止 1~14），暴力枚举所有课程子集；对每个子集按截止日排序后逐门累计时长，任何一门累计超过自己的截止就判该子集不可行；在所有可行子集里取最大规模，和 `schedule_course` 对比。它验证的是"数量最大"这一定义本身。
- IPO 裁判（60 组）：每组 5 个项目，利润 0~5、门槛 0~7 随机。`brute` 是递归穷举：每一层要么收手（返回当前 w），要么枚举每个"未用过且门槛 ≤ w"的项目递归下去，取所有分支终值的最大值。它和 `find_maximized_capital(3,1,p,c)` 对拍。递归里的位掩码 `used|1<<i` 记录已做项目、`used>>i&1` 判断重复，保证每个项目至多做一次——这正是题意的"每项只能一次"。

In [5]:
assert schedule_course([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]) == 3

assert schedule_course([[5, 4]]) == 0

assert find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]) == 4

assert find_maximized_capital(3, 0, [5], [1]) == 0

from itertools import combinations

from random import Random

rng = Random(96)

for _ in range(100):
    a = [(rng.randrange(1, 7), rng.randrange(1, 15)) for _ in range(rng.randrange(8))]
    best = 0
    for k in range(len(a) + 1):
        for group in combinations(a, k):
            total = 0
            valid = True
            for t, d in sorted(group, key=lambda x: x[1]):
                total += t
                if total > d:
                    valid = False
            if valid:
                best = max(best, k)
    assert schedule_course(a) == best

def brute(k, w, p, c, used=0):
    if not k:
        return w
    return max([w] + [brute(k - 1, w + p[i], p, c, used | 1 << i) for i in range(len(p)) if not used >> i & 1 and c[i] <= w])

for _ in range(60):
    p = [rng.randrange(6) for _ in range(5)]
    c = [rng.randrange(8) for _ in range(5)]
    assert find_maximized_capital(3, 1, p, c) == brute(3, 1, p, c)

print('N096: 所有本章断言通过')

N096: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 构造永不撤销选择导致次优的实例。

**练习 2：** 用子集穷举作参照。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（构造"永不撤销"会次优的实例）**：思路是让一门"又长又早截止"的课先占住坑，后面跟两门短课。提示方向：类似 `[[7,7],[2,8],[3,8]]` 的形态——不撤销的贪心会收下 7（7≤7 合法），之后 2 和 3 都因累计超 8 被直接放弃，最终 1 门；而本章算法会在塞第二门时反悔弹掉 7，最终留下 {2,3} 共 2 门。你要写清楚两个策略各自的执行轨迹（哪一步收、哪一步弃），并解释撤销为什么能"以一门换两门"。验证可以直接跑本章接口对出 2，再手写一个"塞不下就跳过"的无撤销版本对出 1。

**练习 2（用子集穷举作参照）**：思路是把 “自动测试”部分 里课程裁判的方法论完整复述并推广。提示方向：穷举所有子集，对每个子集给出可行性判定（怎么排序、累计、和截止比较——想想为什么必须按截止日排序后再累计），取最大规模作为标准答案；然后拿它对拍贪心，既覆盖正常组，也特意构造含"贴线截止"（累计恰好等于截止日，比如 [2,6]+[2,6] 这类）和"单门就超期"的边界组。写输入输出时注意空列表（0 门课，答案 0）这个边界也要进对拍。IPO 侧如果想加练，可以把 `brute` 的递归改写成显式的"选或不选"枚举，验证两者一致。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from heapq import heappush, heappop

def schedule_course(courses):
    total = 0
    chosen = []
    for duration, deadline in sorted(courses, key=lambda c: c[1]):
        heappush(chosen, -duration)
        total += duration
        if total > deadline:
            total += heappop(chosen)
    return len(chosen)

def find_maximized_capital(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    i = 0
    available = []
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heappush(available, -projects[i][1])
            i += 1
        if not available or available[0] >= 0:
            break
        w -= heappop(available)
    return w

# 示例与回归测试
assert schedule_course([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]) == 3

assert schedule_course([[5, 4]]) == 0

assert find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]) == 4

assert find_maximized_capital(3, 0, [5], [1]) == 0

from itertools import combinations

from random import Random

rng = Random(96)

for _ in range(100):
    a = [(rng.randrange(1, 7), rng.randrange(1, 15)) for _ in range(rng.randrange(8))]
    best = 0
    for k in range(len(a) + 1):
        for group in combinations(a, k):
            total = 0
            valid = True
            for t, d in sorted(group, key=lambda x: x[1]):
                total += t
                if total > d:
                    valid = False
            if valid:
                best = max(best, k)
    assert schedule_course(a) == best

def brute(k, w, p, c, used=0):
    if not k:
        return w
    return max([w] + [brute(k - 1, w + p[i], p, c, used | 1 << i) for i in range(len(p)) if not used >> i & 1 and c[i] <= w])

for _ in range(60):
    p = [rng.randrange(6) for _ in range(5)]
    c = [rng.randrange(8) for _ in range(5)]
    assert find_maximized_capital(3, 1, p, c) == brute(3, 1, p, c)

print('N096: 所有本章断言通过')

N096: 所有本章断言通过


## 11. 代表题与迁移

- **630. Course Schedule III（canonical）**：这题就是 `schedule_course` 的原题——截止日约束下最多修几门课。它练的是本章的核心招式：**按截止期排序 + 超时弹掉已选最长者**，也就是"堆让贪心学会反悔"。
- **502. IPO（canonical）**：这题就是 `find_maximized_capital` 的原题——资本门槛下依次挑最大利润。它练的是"排序推进解锁 + 堆取最优"的双结构配合，以及"资源单调增长所以贪心安全"的判断。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。